## Parses Option Data From Excel Sheet And Derives Spot Price + ATM Volatility

After parsing, data is stored in IV_Out/ for consumption by Butterfly.py. IV_Out also contains a plots folder that holds images of intraday atm volality, atm skew and atm curvature.

In [1]:
import utils.IV as iv

iv.main(
    options_data_path="Option_Data.xlsx",
    INPUT_DIR=".",
    OUTPUT_DIR="IV_Out",
    k_window=0.15,
)


Reading workbook sheets:   0%|          | 0/121 [00:00<?, ?it/s]

Inferring spot from parity:   0%|          | 0/66051 [00:00<?, ?it/s]

[TIMER] Initial quote panel + spot merge + 1DTE filtering: 383.81s
[TIMER] Minute sync complete: 69.88s


Computing IVs:   0%|          | 0/514182 [00:00<?, ?it/s]

[TIMER] IVs and vegas computed: 19.22s

VALIDATION REPORT

[1] BASIC DATA CHECKS
Rows in q: 514182
Unique expiries: [datetime.date(2026, 1, 23), datetime.date(2026, 1, 27), datetime.date(2026, 1, 28), datetime.date(2026, 1, 29), datetime.date(2026, 1, 30), datetime.date(2026, 2, 3), datetime.date(2026, 2, 4), datetime.date(2026, 2, 5), datetime.date(2026, 2, 6), datetime.date(2026, 2, 10), datetime.date(2026, 2, 11), datetime.date(2026, 2, 12), datetime.date(2026, 2, 13), datetime.date(2026, 2, 18), datetime.date(2026, 2, 19), datetime.date(2026, 2, 20), datetime.date(2026, 2, 24), datetime.date(2026, 2, 25), datetime.date(2026, 2, 26), datetime.date(2026, 2, 27), datetime.date(2026, 3, 3), datetime.date(2026, 3, 4), datetime.date(2026, 3, 5), datetime.date(2026, 3, 6)]
Unique option types: ['C', 'P']
Strike range: 640.0 to 730.0
Price range: 0.01 to 57.75
Volume range: 1.0 to 59632.0
            timestamp expiry_date cp      K   mid  volume
0 2026-01-22 09:30:00  2026-01-23  P  646.0 

Fitting surface features:   0%|          | 0/9766 [00:00<?, ?it/s]

DROP REASONS: {'empty_after_iv': 0, 'too_few_pts_loc': 0, 'too_few_left_right': 0, 'total_unique_fail': 0, 'left_unique_fail': 0, 'right_unique_fail': 0, 'left_span_fail': 0, 'right_span_fail': 0, 'iv_cap_fail': 0, 'fit_fail': 0, 'curvature_support_fail': 2633, 'kept': 9766}
[TIMER] Feature fitting complete: 92.68s

FEATURE VALIDATION REPORT

[1] FEATURE COVERAGE
            has_atm  has_skew  has_curvature
dte_bucket                                  
1DTE            1.0       1.0       0.730391

[2] FEATURE SUMMARY
         sigma_atm         skew    curvature   num_quotes
count  9766.000000  9766.000000  7133.000000  9766.000000
mean      0.180441    -3.275540    10.352476    34.096457
std       0.046749     0.735077    36.808117     6.269532
min       0.092440    -5.421531   -79.743642    14.000000
25%       0.145635    -3.812656   -17.015881    30.000000
50%       0.179419    -3.359122    10.727998    34.000000
75%       0.211707    -2.735028    39.065473    39.000000
max       0.31

## 

In [2]:
import utils.Butterfly as butterfly

butterfly.main(INPUT_DIR="IV_Out", OUTPUT_DIR="Butterfly_Out")

Top day summary:
trade_date  n_obs  n_events  max_event_score  sigma_range  skew_range  curvature_range
2026-01-28    391        10        20.658792     0.028444    2.663329       683.819564
2026-02-03    391        15        17.875404     0.101012    3.349172       741.381739
2026-01-27    391        17        17.296564     0.018640    2.117571       346.890606
2026-02-04    391        16        16.719057     0.081220    3.911756       731.387836
2026-02-18    391        21        16.589134     0.086372    2.527142       534.712690
2026-02-23    391        26        16.476229     0.058461    2.422420       491.205325
2026-02-17    391        17        16.063009     0.086561    4.010358       369.715857
2026-02-11    391        22        15.522630     0.072327    2.436755       528.939727
2026-02-25    391        10        15.085869     0.019076    1.356256       320.171019
2026-01-26    391        19        14.476857     0.038483    3.089212       484.171483

Selected intraday day: 20

##

In [3]:
import utils.Regime_Label as regime

regime.main(
    INPUT_DIR="Butterfly_Out",
    OUTPUT_DIR="Regime_Out",
    features_file="option_b_intraday_features_events_1dte.csv",
)

Regime fit summary:
       feature  regime regime_name  n_obs     alpha     beta        mu  lambda_per_min  half_life_min  shock_std       r2
curvature_bfly       0        slow   5372  6.690288 0.900813 67.451273        0.104458       6.635682  66.534909 0.796691
curvature_bfly       1        fast   3988  6.710755 0.808454 35.034699        0.212631       3.259853  80.937920 0.667707
     sigma_atm       0        slow   5372  0.000277 0.998388  0.171955        0.001614     429.514817   0.002507 0.996912
     sigma_atm       1        fast   3988  0.000396 0.997179  0.140354        0.002825     245.346142   0.003231 0.995636
          skew       0        slow   5372 -0.494040 0.849764 -3.288435        0.162796       4.257759   0.332181 0.702465
          skew       1        fast   3988 -0.794297 0.757411 -3.274246        0.277850       2.494685   0.445355 0.583001

Fast vs slow comparison:
       feature  slow_lambda_per_min  fast_lambda_per_min  lambda_ratio_fast_to_slow  slow_half_life_